In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_model import reccurentActor
from torch_learning_utils import generateEpisodeWorker, generateSeveralEpisodes, generateSeveralEpisodesParallel
import time

In [2]:
model = reccurentActor()
model.set_initial_state('actor_weights.pth')

c:\Proga\leverege_simul\for_server\model_train\torch_model.py:120: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(file_path, map_location = device)


In [3]:
WIDTH = 0.4
TOP_LEVERAGE = 2 + WIDTH/2
BOTTOM_LEVERAGE = 2 - WIDTH/2

df = pd.read_csv('wbtc_train_data_expanded.csv')
nav = df['nav'].values
nect_price = df['nect_price'].values
vol_token_price = df['wbtc_price'].values
volatile_part = df['wbtc_frac'].values

nav = nav[::4]
nect_price = nect_price[::4]
vol_token_price = vol_token_price[::4]
volatile_part = volatile_part[::4]

default_configs = {
    'nect_price': nect_price,
    'nav': nav,
    'vol_part': volatile_part,
    'vol_token_honey_price': vol_token_price,
    'is_reversed': True,
    'top_leverage': TOP_LEVERAGE,
    'bottom_leverage': BOTTOM_LEVERAGE
}

In [4]:
info = generateEpisodeWorker(
    model_state=model.state_dict(),
    configs=default_configs,
    seed=100,
    epsilon=0.03,
    show_setup=False
)
print(info['reward'])
print(info['mech_quantile'])
#print(info['agent_quantile'])
#print(np.exp(info['log_probs']))

0.8348211603684077
0.8348211603684077
-6.4739394024033
0.4239667818028969


In [5]:
start = time.time()
generateSeveralEpisodes(
    model_state=model.state_dict(), 
    configs=[default_configs],
    epsilon=0.05,
    n_envs=6
    )
end = time.time()
print(end - start)

action scale 1
primordial penalty 0.25
penalty damping 0.9
reward scale 1
reward bound 5
loss scale 0
liquidation penalty 0
risk coef 0
boundary reward scale 0
idle reward 0.05
rebalancing reward 0.2
extravagance penalty 0.1
0.884947798280141
0.884947798280141
0.04418287077944971
0.04418287077944971
0.5922602280825601
0.5922602280825601
0.6211336852273778
0.6211336852273778
0.5570708921857715
0.5570708921857715
0.1053543275986425
0.1053543275986425
73.27656769752502


In [ ]:
start = time.time()
results = generateSeveralEpisodesParallel(
    model_state=model.state_dict(), 
    configs=[default_configs],
    epsilon=0.05
)
end = time.time()
print(end - start)

21.02359676361084


In [ ]:
ep = results[0]

obs_np = ep["obs"]
actions_np = ep["actions"] 
reward = ep["reward"]
quantile = ep["agent_quantile"]

print(quantile)
print(reward)
obs_t = torch.tensor(obs_np, dtype=torch.float32, device = 'cpu').unsqueeze(0)
actions_t = torch.tensor(actions_np, dtype=torch.long, device = 'cpu') 
hidden = model.init_hidden(batch_size=1, device = 'cpu')

logits, _ = model.forward(obs_t, hidden)         
print(logits[0])
dist = torch.distributions.Categorical(logits=logits[0])
log_probs = dist.log_prob(actions_t)              
entropies = dist.entropy().mean()

print(log_probs)
print(entropies)

advantage = reward - 0.01
policy_loss = -advantage * log_probs.mean()
entropy_loss = -0.001 * entropies

total_loss = policy_loss + entropy_loss
batch_loss = total_loss

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer.zero_grad()
batch_loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
optimizer.step()

0.3600570723170092
-7.922567106506062
tensor([[ 9.0730,  1.4436, -3.6304, -1.7666],
        [ 9.2233,  1.4782, -3.6609, -1.8138],
        [ 9.2160,  1.4767, -3.6554, -1.8132],
        ...,
        [ 9.1666,  1.4549, -3.6297, -1.8006],
        [ 9.1254,  1.4486, -3.6118, -1.7958],
        [ 9.1622,  1.4560, -3.6235, -1.8055]], grad_fn=<SelectBackward0>)
tensor([-7.6299e+00, -4.5109e-04, -4.5395e-04,  ..., -4.6730e-04,
        -4.8447e-04, -4.7016e-04], grad_fn=<SqueezeBackward1>)
tensor(0.0132, grad_fn=<MeanBackward0>)
